# GPT-OSS Colab + Google Drive Bootstrap

This notebook prepares a persistent, Colab-only GPT-OSS workspace. It does not use the Open Assistant mobile app. The notebook never deletes existing Drive data and does not download model weights unless its explicit storage and hardware gates pass.

In [ ]:
# User-controlled policy switches. Keep model transfer disabled during the first audit.
REQUESTED_MODEL = 'openai/gpt-oss-120b'
ALLOW_MODEL_DOWNLOAD = False
ALLOW_GPT_OSS_20B_FALLBACK = False
WORKSPACE_ROOT = '/content/drive/MyDrive/AI_Assistant/GPT_OSS_Colab'
MIN_VRAM_120B_GIB = 80
MIN_FREE_DRIVE_120B_GIB = 150
MIN_VRAM_20B_GIB = 16
MIN_FREE_DRIVE_20B_GIB = 40
print('Policy loaded. Model downloads are disabled:', not ALLOW_MODEL_DOWNLOAD)

In [ ]:
# This opens only Google's normal Colab Drive authorization flow.
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('Drive mount request complete.')

In [ ]:
from pathlib import Path
import json, platform, shutil, subprocess, sys
from datetime import datetime, timezone

root = Path(WORKSPACE_ROOT)
folders = [
    'models', 'model_cache', 'manifests', 'notebooks', 'logs', 'configs',
    'documents', 'knowledge_base', 'vector_db', 'memory', 'checkpoints'
]
for folder in folders:
    (root / folder).mkdir(parents=True, exist_ok=True)
print(f'Workspace ready: {root}')
print('Created/reused:', ', '.join(folders))

In [ ]:
def run_text(command):
    result = subprocess.run(command, shell=True, text=True, capture_output=True)
    return {'returncode': result.returncode, 'stdout': result.stdout.strip(), 'stderr': result.stderr.strip()}

gpu_probe = run_text('nvidia-smi --query-gpu=name,memory.total,memory.free,driver_version --format=csv,noheader')
disk = shutil.disk_usage(root)
torch_probe = run_text("python -c \"import torch; print(torch.__version__); print(torch.cuda.is_available()); print(torch.version.cuda)\"")
report = {
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'python': sys.version,
    'platform': platform.platform(),
    'workspace_root': str(root),
    'drive_free_bytes': disk.free,
    'drive_total_bytes': disk.total,
    'gpu_probe': gpu_probe,
    'torch_probe': torch_probe,
    'requested_model': REQUESTED_MODEL,
    'model_download_enabled': ALLOW_MODEL_DOWNLOAD
}
report_path = root / 'manifests' / 'environment_report.json'
report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
print(json.dumps(report, indent=2))
print(f'Persisted: {report_path}')

In [ ]:
# Parse GPU memory conservatively from the nvidia-smi probe.
def gpu_memory_gib(probe):
    if probe['returncode'] != 0 or not probe['stdout']:
        return 0.0
    first_line = probe['stdout'].splitlines()[0]
    fields = [value.strip() for value in first_line.split(',')]
    for field in fields:
        if 'MiB' in field:
            try:
                return float(field.split()[0]) / 1024
            except (ValueError, IndexError):
                continue
    return 0.0

drive_free_gib = disk.free / (1024 ** 3)
vram_gib = gpu_memory_gib(gpu_probe)
can_run_120b = vram_gib >= MIN_VRAM_120B_GIB and drive_free_gib >= MIN_FREE_DRIVE_120B_GIB
can_run_20b = vram_gib >= MIN_VRAM_20B_GIB and drive_free_gib >= MIN_FREE_DRIVE_20B_GIB

if can_run_120b:
    selected_model = 'openai/gpt-oss-120b'
    selection_reason = '80GB GPU and conservative Drive capacity gates passed.'
elif ALLOW_GPT_OSS_20B_FALLBACK and can_run_20b:
    selected_model = 'openai/gpt-oss-20b'
    selection_reason = '120B gate failed; user-approved 20B fallback gate passed.'
else:
    selected_model = None
    selection_reason = 'No model selected. Enable an explicitly approved fallback only after reviewing the persisted report.'

gate = {
    'vram_gib': round(vram_gib, 2),
    'drive_free_gib': round(drive_free_gib, 2),
    'can_run_gpt_oss_120b': can_run_120b,
    'can_run_gpt_oss_20b': can_run_20b,
    'selected_model': selected_model,
    'selection_reason': selection_reason
}
gate_path = root / 'manifests' / 'model_feasibility_gate.json'
gate_path.write_text(json.dumps(gate, indent=2), encoding='utf-8')
print(json.dumps(gate, indent=2))

In [ ]:
# Persistent registry. It records intent and never claims that a model is downloaded or runnable before validation.
registry_path = root / 'manifests' / 'model_registry.json'
registry = {
    'requested_model': REQUESTED_MODEL,
    'selected_model': selected_model,
    'license': 'Apache-2.0',
    'runtime': 'undecided_pending_hardware_gate',
    'drive_path': str(root / 'models' / (selected_model or 'not-selected').replace('/', '__')),
    'cache_path': str(root / 'model_cache'),
    'download_status': 'disabled_by_policy' if not ALLOW_MODEL_DOWNLOAD else 'not_started',
    'validation_status': 'not_run',
    'feasibility_gate': str(gate_path),
    'updated_at': datetime.now(timezone.utc).isoformat()
}
registry_path.write_text(json.dumps(registry, indent=2), encoding='utf-8')
print(f'Persisted: {registry_path}')
print(json.dumps(registry, indent=2))

In [ ]:
# Health check is intentionally truthful: model status is PASS only after a real load and inference validation.
health = {
    'drive': 'PASS' if root.exists() else 'FAIL',
    'storage': 'PASS' if drive_free_gib >= MIN_FREE_DRIVE_20B_GIB else 'FAIL',
    'gpu': 'PASS' if vram_gib > 0 else 'FAIL',
    'cuda': 'PASS' if torch_probe['returncode'] == 0 and 'True' in torch_probe['stdout'] else 'FAIL',
    'model_feasibility_120b': 'PASS' if can_run_120b else 'BLOCKED',
    'model_download': registry['download_status'],
    'inference': 'NOT_RUN',
    'rag': 'NOT_CONFIGURED',
    'persistent_memory': 'WORKSPACE_READY',
    'configuration': 'PASS'
}
health_path = root / 'manifests' / 'health_check.json'
health_path.write_text(json.dumps(health, indent=2), encoding='utf-8')
print(json.dumps(health, indent=2))
print(f'Persisted: {health_path}')

## Optional download step

Do not enable this step until the feasibility report shows an 80GB GPU and sufficient free Drive capacity for GPT-OSS-120B. The public model does not require a token for its public metadata audit. If the Hub requests authentication for a selected artifact, authenticate through the standard Hugging Face flow and never paste a token into a shared notebook or repository.